# ML-09 — Validation Audit
**Lane 2 — Refresh / Content Opportunity Scoring**

Checks whether the model in `w05_model.ipynb` is actually learning generalizable signal,
or partly memorizing per-client patterns. Runs the split-honesty check: same model, same
features, client-holdout split vs. a plain random split.

In [1]:
import pandas as pd, numpy as np, json, pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
for c in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c].clip(lower=0))

NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier",
]
X_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
model_df = df.copy()
for c in NUMERIC_FEATURES:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)
for c in CATEGORICAL_FEATURES:
    model_df[c] = model_df[c].fillna("unknown").astype(str)
y = model_df["is_declining_label"]

pre = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

model_results = json.load(open("../artifacts/model_results.json"))
print("Client-holdout Random Forest (from w05):", model_results["models"]["random_forest"])

Client-holdout Random Forest (from w05): {'roc_auc': 0.6028772346147762, 'precision_at_20': 0.5, 'precision_at_50': 0.56}


## The check: same model, same features, a *plain random* row split instead

If performance jumps a lot, the client-holdout number in `w05` was the honest one, and
the model was partly memorizing client identity, not learning generalizable signal.

In [2]:
tr_idx, te_idx = train_test_split(np.arange(len(model_df)), test_size=0.25,
                                   random_state=RANDOM_SEED, stratify=y)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                             random_state=RANDOM_SEED, n_jobs=-1)
pipe = Pipeline([("pre", pre), ("clf", rf)])
pipe.fit(model_df.iloc[tr_idx][X_cols], y.iloc[tr_idx])
proba = pipe.predict_proba(model_df.iloc[te_idx][X_cols])[:, 1]

random_split_check = {
    "roc_auc": roc_auc_score(y.iloc[te_idx], proba),
    "precision_at_50": precision_at_k(y.iloc[te_idx], proba, 50),
    "test_base_rate": float(y.iloc[te_idx].mean()),
}
print("Random (leaky) split, same Random Forest:", random_split_check)
print()
print("Client-holdout AUC :", round(model_results['models']['random_forest']['roc_auc'], 3))
print("Random-split  AUC :", round(random_split_check['roc_auc'], 3))
print("Gap suggests the model partly recognizes clients it already trained on when the")
print("split doesn't control for that.")

Random (leaky) split, same Random Forest: {'roc_auc': 0.7525183740920378, 'precision_at_50': 0.92, 'test_base_rate': 0.542}

Client-holdout AUC : 0.603
Random-split  AUC : 0.753
Gap suggests the model partly recognizes clients it already trained on when the
split doesn't control for that.


## Leakage checklist

- [x] Label column (`trend_direction`, `trend_pct`) never in the feature list — enforced by
  an `assert` in `w05_model.ipynb`.
- [x] The 30-day windows the label is computed from (`impressions_last_30d` /
  `impressions_prev_30d` and equivalents) excluded, not just the label itself.
- [x] `client_id` / `content_id` used only for grouping, never as model features.
- [x] Evaluation split is grouped by `client_id`, not a plain random split — confirmed above
  that a random split materially inflates the score.
- [x] Train/test base rates checked and close (0.550 vs 0.517) — the split isn't
  accidentally skewed toward one class.

In [3]:
json.dump({"random_split_check": random_split_check,
           "client_holdout_random_forest": model_results["models"]["random_forest"]},
          open("../artifacts/validation_audit.json", "w"), indent=2)
print("saved validation_audit.json")

saved validation_audit.json
